In [2]:
!python -m pip install -U finance-datareader pandas requests lxml beautifulsoup4 tqdm

  Attempting uninstall: requests
    Found existing installation: requests 2.34.1
    Uninstalling requests-2.34.1:
      Successfully uninstalled requests-2.34.1



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import FinanceDataReader as fdr 
import pandas as pd 
from tqdm.auto import tqdm 
import time 
 
# Extract stock codes from the disclosure list
df = pd.read_csv("dart_earnings_disclosure_list_2019_2026.csv", dtype=str) 
stock_codes = df["stock_code"].dropna().unique() 
 
print(f"수집 대상 종목 수: {len(stock_codes)}") 
 
# Set the full data collection period
# Start six months before the first disclosure date to secure the estimation window (t = -120 trading days)
START = "2018-07-01" 
END   = "2026-04-30" 
 
all_prices = [] 
 
for code in tqdm(stock_codes): 
    try: 
        price = fdr.DataReader(code, START, END) 
        if price.empty: 
            continue 
 
        price = price.reset_index() 
        price.columns = [c.lower() for c in price.columns] 
        price["stock_code"] = code 
        all_prices.append(price) 
 
    except Exception as e: 
        print(f"오류 {code}: {e}") 
 
    time.sleep(0.1) 
 
# Combine all stock price data
price_df = pd.concat(all_prices, ignore_index=True) 
 
print(f"수집 완료: {price_df['stock_code'].nunique()}개 종목, {len(price_df)}행") 
print(price_df.head())

수집 대상 종목 수: 947


  0%|          | 0/947 [00:00<?, ?it/s]

수집 완료: 947개 종목, 1657925행
        date   open   high    low  close  volume    change stock_code
0 2018-07-02  62116  63317  61515  61716  103017 -0.004308     285130
1 2018-07-03  61916  62249  60648  61049   72274 -0.010808     285130
2 2018-07-04  60515  62850  60515  61516   75423  0.007650     285130
3 2018-07-05  61382  61982  60181  60448   42546 -0.017361     285130
4 2018-07-06  60448  62583  60181  62583   83536  0.035320     285130


In [3]:
price_df.to_csv("stock_price_2018_2026.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {len(price_df):,}행")

저장 완료: 1,657,925행


## 2. Daily Stock Price and Trading Volume Data Collection

### 2-1. Purpose of Data Collection

Daily stock price and trading volume data for individual stocks were collected to conduct the Event Study.

This study calculated Abnormal Returns (AR) and Cumulative Abnormal Returns (CAR) to measure stock price reactions around disclosure events. For this purpose, daily closing prices, returns, and trading volumes were required for each firm in the disclosure sample.

Trading volume data were also collected to examine abnormal trading activity around disclosure events.

---

### 2-2. Sample Selection

The stocks included in the data collection were identified from the previously constructed earnings disclosure dataset:

```text
dart_earnings_disclosure_list_2019_2026.csv
```

Stock codes for the firms in the sample were extracted using the `stock_code` variable, and duplicate stock codes were removed.

```python
stock_codes = df["stock_code"].dropna().unique()
```

Therefore, daily stock price and trading volume data were collected only for KOSPI- and KOSDAQ-listed firms with earnings disclosures included in the analysis.

---

### 2-3. Collection Period

The stock price data were collected over the following period:

```text
2018-07-01 ~ 2026-04-30
```

Although the disclosure sample begins in January 2019, approximately 120 trading days of pre-event data were required to estimate the Market Model for the Event Study.

Therefore, data collection began on July 1, 2018, approximately six months before the first disclosure period.

The Event Study windows were defined as follows:

| Window | Period |
|---|---|
| Estimation window | t = -120 to -21 |
| Event window | t = -1 to +5 |
| Data collection start date | 2018-07-01 |
| Data collection end date | 2026-04-30 |

---

### 2-4. Data Collection Method

Daily stock price and trading volume data for individual stocks were collected using the Python `FinanceDataReader` package.

For each stock code, `fdr.DataReader()` was executed to retrieve daily market data over the specified period. The resulting stock-level datasets were then combined into a single DataFrame.

The data collection procedure consisted of the following steps:

```text
1. Extract stock codes from the disclosure list
2. Collect daily stock price data for each stock
3. Exclude empty datasets
4. Add the stock code variable
5. Combine all stock-level data into a single DataFrame
6. Save the resulting dataset as a CSV file
```

The main variables collected were:

```text
date
open
high
low
close
volume
change
stock_code
```

The `change` variable was used as the daily stock return, while `volume` was used to analyze abnormal trading activity around disclosure events.

---

### 2-5. Data Storage

The final daily stock price and trading volume dataset was saved as:

```text
stock_price_2018_2026.csv
```

The collected variables were used in subsequent analyses as follows:

| Variable | Purpose |
|---|---|
| date | Define event windows based on trading days |
| stock_code | Merge stock price data with disclosure data |
| close | Verify daily stock prices |
| change | Measure individual stock returns |
| volume | Analyze abnormal trading activity |

---

### 2-6. Role in the Analysis

The daily stock price and trading volume data served as core inputs for the Event Study and trading volume analysis.

Specifically, the dataset was used for the following analyses:

```text
1. Calculate abnormal returns around disclosure events
2. Calculate cumulative abnormal returns such as CAR[-1,+1] and CAR[-1,+5]
3. Compare stock price reactions between intraday and after-hours disclosures
4. Test market reactions to after-hours bad-news disclosures
5. Analyze abnormal trading volume around disclosure events
```

Overall, the daily stock price and trading volume dataset provides the core market data required to examine how disclosure timing affects stock prices and trading activity.